# Como a IA prevê a próxima palavra

Nesta aula vamos observar como um modelo de linguagem estima o próximo **token**. A analogia é um corretor ortográfico de celular com esteroides: ele aprendeu padrões e sugere continuações, mas não lê ou entende como uma pessoa.

Texto é dividido em tokens (pedaços de palavras, palavras ou sinais). Cada token corresponde a um ID no vocabulário do modelo. A previsão é uma distribuição estatística condicionada ao texto anterior, não uma prova de compreensão nem de verdade.

## Preparação

Este notebook usa a API HTTP local do Ollama e não baixa modelos por conta própria. Instale e inicie o Ollama, depois execute no terminal:

```bash
uv sync --dev
ollama pull llama3.2:3b
uv run jupyter lab
```

Se o modelo não couber no computador, escolha outro instalado e altere `MODELO` na célula seguinte.

In [ ]:
import html
import math

import matplotlib.pyplot as plt
import requests
from IPython.display import HTML, display

OLLAMA_URL = "http://localhost:11434"
MODELO = "llama3.2:3b"

def post_ollama(endpoint, payload, timeout=120):
    try:
        response = requests.post(
            f"{OLLAMA_URL}{endpoint}", json=payload, timeout=timeout
        )
        response.raise_for_status()
    except requests.exceptions.ConnectionError as exc:
        raise RuntimeError(
            "Não foi possível conectar ao Ollama. Inicie o serviço local e tente novamente."
        ) from exc
    except requests.exceptions.HTTPError as exc:
        detalhe = response.text[:500]
        raise RuntimeError(
            f"Ollama respondeu com HTTP {response.status_code}: {detalhe}"
        ) from exc
    except requests.exceptions.Timeout as exc:
        raise RuntimeError("O Ollama demorou demais para responder.") from exc
    return response.json()

print(f"Servidor: {OLLAMA_URL} | Modelo: {MODELO}")

## 1. Texto vira tokens

Vamos pedir ao tokenizer do próprio modelo para dividir `O gato roeu a roupa`. A API `/api/tokenize` retorna IDs reais; se o servidor não oferecer esse endpoint, a célula explica o problema em vez de inventar uma tokenização.

In [ ]:
frase = "O gato roeu a roupa"

try:
    resultado_tokens = post_ollama(
        "/api/tokenize", {"model": MODELO, "prompt": frase}
    )
    ids = resultado_tokens.get("tokens", [])
    if not ids:
        raise RuntimeError("A API não retornou IDs de token.")

    blocos = []
    for indice, token_id in enumerate(ids):
        try:
            decodificado = post_ollama(
                "/api/detokenize",
                {"model": MODELO, "tokens": [token_id]},
            ).get("content", "")
        except RuntimeError:
            decodificado = ""
        rotulo = decodificado if decodificado else f"ID {token_id}"
        cor = "#d9f0e5" if indice % 2 == 0 else "#fce3c5"
        blocos.append(
            f"<span style='display:inline-block;margin:4px;padding:10px;"
            f"background:{cor};border:1px solid #777;border-radius:4px'>"
            f"{html.escape(rotulo)}<br><small>ID: {html.escape(str(token_id))}</small></span>"
        )
    display(HTML("<p>" + "".join(blocos) + "</p>"))
    print("Texto original:", frase)
    print("Observação: decodificar um token isoladamente pode diferir do contexto ao redor.")
except RuntimeError as erro:
    print(f"Tokenização indisponível: {erro}")
    print("Confira se o Ollama está ativo e atualizado; a API /api/tokenize depende da versão.")

## 2. A próxima aposta

A API de geração pode devolver o token escolhido e os principais candidatos. Usaremos `num_predict=1` para parar depois de um token, `temperature=0.0` para uma amostra determinística e `logprobs` para solicitar os logaritmos das probabilidades. Nem todas as versões/combinações de servidor e modelo oferecem esses dados.

A porcentagem abaixo é `exp(logprob) × 100`. O top 5 é apenas uma parte do vocabulário, então as porcentagens **não** serão normalizadas para somar 100%.

In [ ]:
def pedir_proximo(prompt, temperature=0.0, top_k=5):
    return post_ollama(
        "/api/generate",
        {
            "model": MODELO,
            "prompt": prompt,
            "stream": False,
            "options": {"num_predict": 1, "temperature": temperature},
            "logprobs": True,
            "top_logprobs": top_k,
        },
    )


def ler_candidatos(resultado):
    passos = resultado.get("logprobs") or []
    if isinstance(passos, dict):
        passos = [passos]
    if not passos:
        raise RuntimeError(
            "A resposta não trouxe logprobs. Atualize o Ollama e confira o suporte "
            "do modelo aos campos logprobs/top_logprobs. Nenhum dado foi estimado."
        )

    passo = passos[-1]
    candidatos_brutos = passo.get("top_logprobs", [])
    if isinstance(candidatos_brutos, dict):
        candidatos = [
            {"token": token, "logprob": valor}
            for token, valor in candidatos_brutos.items()
        ]
    else:
        candidatos = candidatos_brutos

    token_escolhido = passo.get("token") or resultado.get("response", "")
    if not candidatos and "logprob" in passo and token_escolhido:
        candidatos = [{"token": token_escolhido, "logprob": passo["logprob"]}]

    linhas = []
    for candidato in candidatos:
        if not isinstance(candidato, dict):
            continue
        token = candidato.get("token", "")
        valor = candidato.get("logprob")
        if valor is not None:
            logprob = float(valor)
            linhas.append(
                {"token": token, "logprob": logprob, "chance": math.exp(logprob)}
            )

    if not linhas:
        raise RuntimeError("Formato de logprobs não reconhecido nesta resposta do Ollama.")
    return token_escolhido, linhas


def mostrar_grafico(linhas, titulo, eixo=None):
    ordenadas = sorted(linhas, key=lambda linha: linha["chance"])
    tokens = [linha["token"] for linha in ordenadas]
    chances = [linha["chance"] * 100 for linha in ordenadas]
    if eixo is None:
        _, eixo = plt.subplots(figsize=(8, max(2.5, len(linhas) * 0.55)))
    eixo.barh(tokens, chances, color="#397d64")
    eixo.set_xlabel("Probabilidade do modelo (%)")
    eixo.set_title(titulo)
    eixo.set_xlim(left=0)
    return eixo


prompt_proverbio = "Água mole em pedra dura, tanto bate até que"
try:
    resposta = pedir_proximo(prompt_proverbio)
    escolhido, candidatos = ler_candidatos(resposta)
    print(f"Prompt: {prompt_proverbio!r}")
    print(f"Continuação escolhida: {escolhido!r}")
    mostrar_grafico(candidatos, "Candidatos para o próximo token")
    plt.tight_layout()
except RuntimeError as erro:
    print(f"Não foi possível mostrar probabilidades: {erro}")

## 3. Geração passo a passo

Agora repetimos a chamada seis vezes e acrescentamos cada token ao prompt. Verde indica probabilidade acima de 80%, amarelo a partir de 50% e vermelho abaixo. Isso é probabilidade atribuída pelo modelo, não certeza factual; tokens também podem ser fragmentos de palavras.

In [ ]:
def gerar_passo_a_passo(prompt, quantidade=6):
    texto = prompt
    historico = []

    for _ in range(quantidade):
        resultado = pedir_proximo(texto, temperature=0.0, top_k=5)
        token, candidatos = ler_candidatos(resultado)
        escolhido = next(
            (linha for linha in candidatos if linha["token"] == token),
            candidatos[0],
        )
        chance = escolhido["chance"]
        historico.append({"token": token, "chance": chance})
        texto += token

    elementos = []
    for linha in historico:
        chance = linha["chance"]
        cor = "#d9f0e5" if chance > 0.8 else "#fff0bd" if chance >= 0.5 else "#f6d4d4"
        elementos.append(
            f"<span style='background:{cor};padding:6px;margin:3px;border-radius:3px'>"
            f"{html.escape(linha['token'])} ({chance:.1%})</span>"
        )

    display(HTML(" → ".join(elementos)))
    print("Texto completo:", texto)
    return historico


try:
    historico_tokens = gerar_passo_a_passo("O gato", quantidade=6)
except RuntimeError as erro:
    print(f"Geração passo a passo indisponível: {erro}")

## 4. Temperatura: mudar a distribuição

Compare o mesmo prompt com temperatura `0.0` e `1.2`. A temperatura altera a distribuição usada na amostragem, mas não cria entendimento nem criatividade humana. Uma execução isolada não garante que a escolha vá mudar.

In [ ]:
figura, eixos = plt.subplots(1, 2, figsize=(12, 4))
comparacoes = [(0.0, eixos[0]), (1.2, eixos[1])]
algum_grafico = False

for temperatura, eixo in comparacoes:
    try:
        resultado = pedir_proximo(prompt_proverbio, temperature=temperatura)
        escolhido, candidatos = ler_candidatos(resultado)
        mostrar_grafico(
            candidatos,
            f"Temperatura {temperatura:.1f} | escolhido: {escolhido!r}",
            eixo=eixo,
        )
        algum_grafico = True
    except RuntimeError as erro:
        eixo.set_title(f"Temperatura {temperatura:.1f}")
        eixo.text(0.5, 0.5, str(erro), ha="center", va="center", wrap=True)
        eixo.set_axis_off()

if algum_grafico:
    plt.tight_layout()
else:
    plt.close(figura)
    print("A comparação requer que o Ollama retorne logprobs para as duas chamadas.")

## 5. Fechamento

O modelo estima quais tokens combinam estatisticamente com a sequência anterior, com base nos padrões aprendidos durante o treinamento. Ele não conhece o significado das palavras como uma pessoa e pode errar mesmo quando atribui alta probabilidade.

**Para conversar:**
- O que mudou ao alterar a temperatura?
- Por que as probabilidades do top 5 não somam necessariamente 100%?
- Uma continuação provável é sempre verdadeira ou adequada?
- Por que olhar para tokens, em vez de palavras inteiras, muda nossa interpretação do processo?